# Fast credit-card fraud training — one model per cell

The old notebook performed **52 fits** (48 CV fits + 4 refits). This notebook defaults
to **3 fits**, no grid search, and early stopping for boosting. XGBoost uses CUDA when available;
Logistic Regression and LightGBM use CPU. The full dataset is retained.

Each model has its own cell and checkpoint. Run top to bottom; use `RESUME = True`
with the **same RUN_NAME and settings** to reuse completed models after interruption.
Only the active unfinished model must restart. Drive persistence is optional.

Outputs: CPU-ready model, evaluation CSVs, plots, dataset fingerprint, dependency versions,
resume bullets, and a ZIP compatible with this project's API/dashboard.
No measured GPU results are pre-filled. Never use synthetic smoke-test metrics on a resume.


## 1. Install once
If packages were already imported, restart the runtime after installation.

In [ ]:
%pip install -q "numpy>=1.26,<3" "pandas>=2.2,<3" "scikit-learn>=1.6,<2" "xgboost>=2.1,<4" "lightgbm>=4.5,<5" "joblib>=1.4,<2" "matplotlib>=3.9,<4" "kagglehub>=0.3,<1"


## 2. Configure
Start with `MODELS = ('xgboost',)` if you want just one model. For the three-model
comparison keep the default. Unselected model cells will skip automatically.
Do not change model choices after seeing test results; use validation for development.

Keep a stable run name when resuming. Use a new name for changed settings.
Enable Drive to preserve checkpoints across Colab disconnects. Authorization may
require the Colab browser rather than the VS Code extension. Without Drive, download
the ZIP before the runtime is deleted. Windows paths are not available on Colab.


In [ ]:
import os
from pathlib import Path

DEVICE = "auto"  # Use CUDA when available; otherwise continue on CPU
MODELS = ("logistic_regression", "xgboost", "lightgbm")
SEED = 42
JOBS = min(2, os.cpu_count() or 1)
MAX_TREES = 300
PATIENCE = 30
MIN_PRECISION = 0.80
DATA_SOURCE = "kaggle"  # or "path"
DATA_PATH = Path("/content/drive/MyDrive/fraud-detection/creditcard.csv")
SAVE_TO_DRIVE = False
RUN_NAME = "fast-01"  # Keep unchanged when resuming this run
RESUME = True

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/fraud-detection/runs")
else:
    OUTPUT_ROOT = Path("/content/fraud-detection/runs")
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
print("Results:", OUTPUT_DIR)


## 3. Load the project training code
The exact local training modules are embedded below so this notebook runs standalone.
This cell only installs project source into a temporary folder; it does not train.
The notebook is generated by `scripts/build_notebook.py` to prevent local/Colab drift.


In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

PROJECT_SOURCES = {'__init__.py': '"""Credit card fraud benchmark and inference tools."""\n', 'data.py': '"""Schema validation and duplicate-safe benchmark splits."""\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nFEATURES = ["Time", *[f"V{i}" for i in range(1, 29)], "Amount"]\nTARGET = "Class"\n\n\ndef validate_features(frame: pd.DataFrame) -> pd.DataFrame:\n    if len(frame) == 0:\n        raise ValueError("At least one transaction is required.")\n    if set(frame.columns) != set(FEATURES) or len(frame.columns) != len(FEATURES):\n        raise ValueError("Expected exactly Time, V1 through V28, and Amount.")\n    if any(not pd.api.types.is_numeric_dtype(frame[c]) or\n           pd.api.types.is_bool_dtype(frame[c]) for c in FEATURES):\n        raise ValueError("All features must be numeric, not boolean.")\n    values = frame[FEATURES].to_numpy(dtype=float)\n    if not np.isfinite(values).all():\n        raise ValueError("Features must contain only finite, non-missing values.")\n    if (frame[["Time", "Amount"]] < 0).any().any():\n        raise ValueError("Time and Amount must be nonnegative.")\n    return frame.loc[:, FEATURES]\n\n\ndef load_dataset(path: str | Path) -> tuple[pd.DataFrame, dict]:\n    frame = pd.read_csv(path)\n    if set(frame.columns) != {*FEATURES, TARGET}:\n        raise ValueError("Training CSV must contain the 30 features and Class only.")\n    validate_features(frame.drop(columns=TARGET))\n    if not frame[TARGET].isin([0, 1]).all() or frame[TARGET].nunique() != 2:\n        raise ValueError("Class must contain both 0 (legitimate) and 1 (fraud).")\n    original_rows = len(frame)\n    # Identical inputs must not cross splits, including conflicting labels.\n    duplicates = frame[frame.duplicated(subset=FEATURES, keep=False)]\n    if not duplicates.empty and duplicates.groupby(FEATURES, dropna=False)[TARGET].nunique().gt(1).any():\n        raise ValueError("Identical feature rows have conflicting Class labels.")\n    frame = frame.drop_duplicates(subset=FEATURES).reset_index(drop=True)\n    frame[TARGET] = frame[TARGET].astype(int)\n    if frame[TARGET].value_counts().min() < 20:\n        raise ValueError("Need at least 20 unique examples of each class for reliable splits.")\n    return frame, {\n        "input_rows": original_rows,\n        "duplicate_rows_removed": original_rows - len(frame),\n        "unique_rows": len(frame),\n        "fraud_count": int(frame[TARGET].sum()),\n        "fraud_prevalence": float(frame[TARGET].mean()),\n    }\n\n\ndef split_dataset(frame: pd.DataFrame, seed: int = 42):\n    train, remainder = train_test_split(\n        frame, test_size=0.30, random_state=seed, stratify=frame[TARGET]\n    )\n    validation, test = train_test_split(\n        remainder, test_size=0.50, random_state=seed, stratify=remainder[TARGET]\n    )\n    return train, validation, test\n', 'training.py': '"""Compare four models without using test data for selection."""\n\nimport hashlib\nimport json\nimport platform\nimport time\nfrom datetime import UTC, datetime\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport joblib\nimport matplotlib\nimport numpy as np\nimport pandas as pd\nfrom lightgbm import LGBMClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    ConfusionMatrixDisplay,\n    PrecisionRecallDisplay,\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_curve,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom sklearn.model_selection import GridSearchCV, StratifiedKFold\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom xgboost import XGBClassifier\n\nfrom fraud_detection.data import FEATURES, TARGET, load_dataset, split_dataset\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n\ndef model_candidates(ratio: float, seed: int, jobs: int, quick: bool):\n    trees = 40 if quick else 200\n    return {\n        "logistic_regression": (\n            Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(\n                max_iter=2000, random_state=seed))]),\n            {"model__C": [0.1, 1.0], "model__class_weight": [None, "balanced"]},\n        ),\n        "random_forest": (\n            Pipeline([("model", RandomForestClassifier(\n                n_estimators=trees, n_jobs=jobs, random_state=seed))]),\n            {"model__max_depth": [8, None], "model__class_weight": [None, "balanced"]},\n        ),\n        "xgboost": (\n            Pipeline([("model", XGBClassifier(\n                n_estimators=trees, learning_rate=0.1, tree_method="hist",\n                eval_metric="logloss", n_jobs=jobs, random_state=seed))]),\n            {"model__max_depth": [3, 6], "model__scale_pos_weight": [1.0, ratio]},\n        ),\n        "lightgbm": (\n            Pipeline([("model", LGBMClassifier(\n                n_estimators=trees, learning_rate=0.1, n_jobs=jobs,\n                random_state=seed, verbosity=-1, deterministic=True, force_col_wise=True))]),\n            {"model__num_leaves": [15, 31], "model__scale_pos_weight": [1.0, ratio]},\n        ),\n    }\n\n\ndef choose_threshold(labels, scores, min_precision: float) -> tuple[float, bool]:\n    precision, recall, thresholds = precision_recall_curve(labels, scores)\n    eligible = np.flatnonzero(precision[:-1] >= min_precision)\n    if len(eligible):\n        # Thresholds are ascending: break equal-recall ties toward fewer alerts.\n        best_recall = recall[eligible].max()\n        index = eligible[recall[eligible] == best_recall][-1]\n        return float(thresholds[index]), True\n    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(\n        precision[:-1] + recall[:-1], np.finfo(float).eps\n    )\n    return float(thresholds[np.argmax(f1)]), False\n\n\ndef evaluate(labels, scores, threshold: float) -> dict:\n    predicted = scores >= threshold\n    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()\n    return {\n        "accuracy": float(np.mean(np.asarray(labels) == predicted)),\n        "average_precision": float(average_precision_score(labels, scores)),\n        "roc_auc": float(roc_auc_score(labels, scores)),\n        "precision": float(precision_score(labels, predicted, zero_division=0)),\n        "recall": float(recall_score(labels, predicted, zero_division=0)),\n        "f1": float(f1_score(labels, predicted, zero_division=0)),\n        "true_negatives": int(tn), "false_positives": int(fp),\n        "false_negatives": int(fn), "true_positives": int(tp),\n        "alert_rate": float(np.mean(predicted)), "threshold": float(threshold),\n    }\n\n\ndef train(data: str | Path, output: str | Path, seed: int = 42,\n          jobs: int = 2, quick: bool = False, min_precision: float = 0.80) -> dict:\n    if not 0 < min_precision <= 1:\n        raise ValueError("Minimum precision must be in (0, 1].")\n    if jobs < 1:\n        raise ValueError("Jobs must be positive.")\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise ValueError("Output directory must be empty; use a new run directory.")\n    frame, data_summary = load_dataset(data)\n    training, validation, test = split_dataset(frame, seed)\n    output.mkdir(parents=True, exist_ok=True)\n    ratio = float((training[TARGET] == 0).sum() / (training[TARGET] == 1).sum())\n    models, validation_rows, search_rows = {}, [], []\n    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)\n    for name, (pipeline, parameters) in model_candidates(ratio, seed, jobs, quick).items():\n        print(f"Training {name} (4 candidates, 3 folds)...", flush=True)\n        start = time.perf_counter()\n        search = GridSearchCV(pipeline, parameters, scoring="average_precision", cv=cv,\n                              n_jobs=1, error_score="raise", refit=True)\n        search.fit(training[FEATURES], training[TARGET])\n        elapsed = time.perf_counter() - start\n        scores = search.best_estimator_.predict_proba(validation[FEATURES])[:, 1]\n        threshold, met = choose_threshold(validation[TARGET], scores, min_precision)\n        validation_rows.append({"model": name, **evaluate(validation[TARGET], scores, threshold),\n                                "precision_target_met": met, "training_seconds": elapsed})\n        models[name] = (search.best_estimator_, threshold)\n        search_rows.append({"model": name, "best_parameters": search.best_params_,\n                            "cv_average_precision": float(search.best_score_),\n                            "candidates": [\n                                {"parameters": p, "mean_cv_average_precision": float(s)}\n                                for p, s in zip(search.cv_results_["params"],\n                                                search.cv_results_["mean_test_score"])]})\n    # Freeze selection before looking at any held-out test predictions.\n    selected = max(validation_rows, key=lambda row: row["average_precision"])["model"]\n    test_rows = []\n    fig, axis = plt.subplots(figsize=(8, 6))\n    for name, (pipeline, threshold) in models.items():\n        pipeline.predict_proba(test[FEATURES].iloc[:min(100, len(test))])\n        start = time.perf_counter()\n        scores = pipeline.predict_proba(test[FEATURES])[:, 1]\n        seconds = time.perf_counter() - start\n        test_rows.append({"model": name, **evaluate(test[TARGET], scores, threshold),\n                          "prediction_seconds": seconds,\n                          "batch_transactions_per_second": len(test) / seconds,\n                          "amortized_ms_per_transaction": seconds * 1000 / len(test)})\n        PrecisionRecallDisplay.from_predictions(test[TARGET], scores, name=name, ax=axis)\n        cm_fig, cm_axis = plt.subplots(figsize=(5, 4))\n        ConfusionMatrixDisplay.from_predictions(\n            test[TARGET], scores >= threshold, labels=[0, 1],\n            display_labels=["Legitimate", "Fraud"], ax=cm_axis, colorbar=False)\n        cm_axis.set_title(name)\n        cm_fig.tight_layout()\n        cm_fig.savefig(output / f"confusion_{name}.png", dpi=140)\n        plt.close(cm_fig)\n    axis.axhline(test[TARGET].mean(), linestyle="--", color="gray", label="Prevalence")\n    axis.set_title("Held-out test precision-recall comparison")\n    axis.legend(loc="best")\n    fig.tight_layout()\n    fig.savefig(output / "precision_recall.png", dpi=140)\n    plt.close(fig)\n    validation_table, test_table = pd.DataFrame(validation_rows), pd.DataFrame(test_rows)\n    validation_table.to_csv(output / "validation_metrics.csv", index=False)\n    test_table.to_csv(output / "test_metrics.csv", index=False)\n    test_table.merge(validation_table[["model", "training_seconds"]], on="model").to_csv(\n        output / "comparison.csv", index=False)\n    timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")\n    pipeline, threshold = models[selected]\n    joblib.dump({"pipeline": pipeline, "threshold": threshold, "features": FEATURES,\n                 "model_name": selected, "version": timestamp}, output / "model.joblib")\n    with Path(data).open("rb") as stream:\n        digest = hashlib.file_digest(stream, "sha256").hexdigest()\n    report = {\n        "version": timestamp, "selected_model": selected,\n        "selection_metric": "validation average_precision",\n        "threshold_policy": "Maximum validation recall at requested minimum precision; "\n                            "fallback to maximum validation F1 if target is unattainable.",\n        "minimum_validation_precision": min_precision,\n        "dataset": {**data_summary, "sha256": digest},\n        "splits": {name: {"rows": len(part), "frauds": int(part[TARGET].sum())}\n                   for name, part in [("train", training), ("validation", validation), ("test", test)]},\n        "seed": seed, "jobs": jobs, "quick": quick,\n        "split_strategy": "stratified random 70/15/15 after exact-feature deduplication",\n        "limitations": ["Random splits do not measure future-transaction generalization.",\n                        "Class weighting can affect probability calibration; scores are not calibrated risks.",\n                        "Precision targets on validation are not guarantees on test or production data.",\n                        "Batch timing is not API latency or a production scalability guarantee."],\n        "environment": {"python": platform.python_version(), "platform": platform.platform(),\n                        "packages": {p: version(p) for p in ["numpy", "pandas", "scikit-learn",\n                                                             "xgboost", "lightgbm", "joblib"]}},\n        "search": search_rows, "validation": validation_rows, "test": test_rows,\n    }\n    (output / "report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(test_table.to_string(index=False))\n    print(f"Selected on validation: {selected}. Artifacts: {output.resolve()}")\n    return report\n', 'fast_training.py': '"""Staged, resumable training: one fit per model, with a separate stopping split."""\n\nimport hashlib\nimport json\nimport os\nimport platform\nimport time\nfrom datetime import UTC, datetime\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom lightgbm import LGBMClassifier, early_stopping\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\nfrom xgboost import XGBClassifier\nfrom xgboost.core import XGBoostError\n\nfrom fraud_detection.data import FEATURES, TARGET, load_dataset, split_dataset\nfrom fraud_detection.training import choose_threshold, evaluate\n\nFAST_MODELS = ("logistic_regression", "xgboost", "lightgbm")\nPACKAGES = ("numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "joblib")\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef dump_model(path, value):\n    temporary = Path(str(path) + ".tmp")\n    joblib.dump(value, temporary)\n    os.replace(temporary, path)\n\n\ndef check_device(device, seed=42):\n    if device not in {"auto", "cpu", "cuda"}:\n        raise ValueError("Device must be auto, cpu or cuda.")\n    if device != "cpu":\n        probe = XGBClassifier(n_estimators=1, max_depth=1, tree_method="hist", device="cuda")\n        try:\n            probe.fit(np.random.default_rng(seed).normal(size=(32, 4)), np.tile([0, 1], 16))\n        except XGBoostError:\n            if device == "cuda":\n                raise\n            print("XGBoost CUDA probe failed; continuing training on CPU.", flush=True)\n            return "cpu"\n        actual = json.loads(probe.get_booster().save_config())["learner"]["generic_param"]["device"]\n        if not actual.startswith("cuda"):\n            if device == "cuda":\n                raise RuntimeError("XGBoost fell back to CPU. Select a GPU runtime or device=\'cpu\'.")\n            print("XGBoost CUDA unavailable; continuing training on CPU.", flush=True)\n            return "cpu"\n        print(f"XGBoost GPU verified: {actual}", flush=True)\n        return "cuda"\n    return "cpu"\n\n\ndef prepare_run(data, output, seed=42, jobs=2, min_precision=0.8,\n                device="cpu", models=FAST_MODELS, max_trees=300, patience=30, resume=False):\n    """Validate once, then reuse the returned state across notebook cells."""\n    if not 0 < min_precision <= 1 or jobs < 1:\n        raise ValueError("Minimum precision must be in (0, 1] and jobs must be positive.")\n    models = tuple(models)\n    if not models or len(set(models)) != len(models) or set(models) - set(FAST_MODELS):\n        raise ValueError(f"Choose unique models from {FAST_MODELS}.")\n    if max_trees < 1 or patience < 1:\n        raise ValueError("max_trees and patience must be positive.")\n    if device not in {"auto", "cpu", "cuda"}:\n        raise ValueError("Device must be auto, cpu or cuda.")\n    if "xgboost" in models:\n        device = check_device(device, seed)\n    elif device == "auto":\n        device = "cpu"\n    start = time.perf_counter()\n    data, output = Path(data), Path(output)\n    with data.open("rb") as stream:\n        digest = hashlib.file_digest(stream, "sha256").hexdigest()\n    # Do not reuse checkpoints after code, data, dependencies, or settings change.\n    source_hash = hashlib.sha256()\n    for name in ("data.py", "training.py", "fast_training.py"):\n        source_hash.update(Path(__file__).with_name(name).read_bytes())\n    packages = {p: version(p) for p in PACKAGES}\n    config = {"seed": seed, "jobs": jobs, "min_precision": min_precision, "device": device,\n                  "models": list(models), "max_trees": max_trees, "patience": patience,\n                  "dataset_sha256": digest, "source_sha256": source_hash.hexdigest(),\n                  "packages": packages, "python": platform.python_version()}\n    manifest_path = output / "run_config.json"\n    if output.exists() and any(output.iterdir()):\n        if not resume:\n            raise ValueError("Output directory must be empty; use --resume or a new run directory.")\n        if not manifest_path.exists():\n            raise ValueError("Cannot resume: run_config.json is missing. Use a new directory.")\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if manifest["config"] != config:\n            raise ValueError("Cannot resume: data, code, settings or versions changed. Use a new directory.")\n    else:\n        manifest = {"version": datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ"), "config": config}\n    print("Loading, validating and deduplicating dataset...", flush=True)\n    frame, summary = load_dataset(data)\n    training, validation, test = split_dataset(frame, seed)\n    # Boosting stopping data is drawn ONLY from the training partition.\n    fit, stopping = train_test_split(training, test_size=0.15, random_state=seed,\n                                    stratify=training[TARGET])\n    output.mkdir(parents=True, exist_ok=True)\n    (output / "checkpoints").mkdir(exist_ok=True)\n    write_json(manifest_path, manifest)\n    state = {"config": config, "output": output, "manifest": manifest, "summary": summary,\n                 "training": training, "validation": validation, "test": test, "fit": fit, "stopping": stopping,\n                 "results": {}, "preparation_seconds": time.perf_counter() - start}\n    print(f"Ready: {len(frame):,} unique rows; {len(test):,} held out. "\n          f"{len(models)} model fits, no grid search.", flush=True)\n    return state\n\n\ndef fit_model(state, name):\n    """One model per call/cell; successful models are checkpointed immediately."""\n    config, output = state["config"], state["output"]\n    if name not in config["models"]:\n        raise ValueError(f"Model {name} was not configured for this run.")\n    checkpoint = output / "checkpoints" / f"{name}.joblib"\n    if checkpoint.exists():\n        result = joblib.load(checkpoint)  # Only resume your own trusted run.\n        state["results"][name] = result\n        print(f"Reused completed checkpoint: {name}", flush=True)\n        return result["validation"]\n    if (output / "report.json").exists():\n        raise ValueError("Completed run cannot be retrained. Use a new directory.")\n    seed, jobs = config["seed"], config["jobs"]\n    fit, stopping, training = state["fit"], state["stopping"], state["training"]\n    validation = state["validation"]\n    common = {"n_estimators": config["max_trees"], "learning_rate": 0.05,\n                  "n_jobs": jobs, "random_state": seed}\n    if name == "logistic_regression":\n        pipeline = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(\n            C=1.0, max_iter=500, tol=1e-3, random_state=seed))])\n    elif name == "xgboost":\n        pipeline = Pipeline([("model", XGBClassifier(\n            **common, tree_method="hist", device=config["device"], max_depth=4,\n            max_bin=128, eval_metric="aucpr", early_stopping_rounds=config["patience"]))])\n    else:\n        pipeline = Pipeline([("model", LGBMClassifier(\n            **common, num_leaves=15, max_bin=127, metric="average_precision",\n            verbosity=-1, deterministic=True, force_col_wise=True))])\n    print(f"Training {name} on {config[\'device\'] if name == \'xgboost\' else \'cpu\'}...", flush=True)\n    start = time.perf_counter()\n    with threadpool_limits(limits=jobs):\n        if name == "logistic_regression":\n            pipeline.fit(training[FEATURES], training[TARGET])\n            best_iteration = None\n        else:\n            options = {"eval_set": [(stopping[FEATURES], stopping[TARGET])]}\n            if name == "xgboost":\n                options["verbose"] = False\n            else:\n                options["callbacks"] = [early_stopping(config["patience"],\n                                                       first_metric_only=True, verbose=False)]\n            pipeline.named_steps["model"].fit(fit[FEATURES], fit[TARGET], **options)\n            estimator = pipeline.named_steps["model"]\n            best_iteration = (int(estimator.best_iteration) + 1 if name == "xgboost"\n                              else int(estimator.best_iteration_))\n    elapsed = time.perf_counter() - start\n    # The same CPU inference configuration is evaluated, exported and used by the API.\n    if name == "xgboost":\n        pipeline.set_params(model__device="cpu")\n    scores = pipeline.predict_proba(validation[FEATURES])[:, 1]\n    threshold, met = choose_threshold(validation[TARGET], scores, config["min_precision"])\n    row = {"model": name, **evaluate(validation[TARGET], scores, threshold),\n           "precision_target_met": met, "training_seconds": elapsed,\n           "best_iteration": best_iteration}\n    result = {"pipeline": pipeline, "threshold": threshold, "validation": row}\n    dump_model(checkpoint, result)\n    state["results"][name] = result\n    print(f"Saved {name}: {elapsed:.2f}s, validation AP={row[\'average_precision\']:.4f}, "\n          f"trees={best_iteration}", flush=True)\n    return row\n\n\ndef finish_run(state):\n    """Freeze validation selection, evaluate test once, export a CPU-ready bundle."""\n    output, config = state["output"], state["config"]\n    if (output / "report.json").exists():\n        print("Run already complete; returning saved metrics without re-evaluating test.")\n        return json.loads((output / "report.json").read_text(encoding="utf-8"))\n    missing = set(config["models"]) - set(state["results"])\n    if missing:\n        raise ValueError(f"Train the remaining models before evaluation: {sorted(missing)}")\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay\n\n    rows = [state["results"][name]["validation"] for name in config["models"]]\n    selected = max(rows, key=lambda row: row["average_precision"])["model"]\n    test = state["test"]\n    test_rows = []\n    fig, axis = plt.subplots(figsize=(8, 6))\n    for name in config["models"]:\n        result = state["results"][name]\n        pipeline, threshold = result["pipeline"], result["threshold"]\n        pipeline.predict_proba(test[FEATURES].iloc[:100])\n        start = time.perf_counter()\n        scores = pipeline.predict_proba(test[FEATURES])[:, 1]\n        seconds = time.perf_counter() - start\n        test_rows.append({"model": name, **evaluate(test[TARGET], scores, threshold),\n                          "prediction_seconds": seconds,\n                          "batch_transactions_per_second": len(test) / seconds,\n                          "amortized_ms_per_transaction": seconds * 1000 / len(test)})\n        PrecisionRecallDisplay.from_predictions(test[TARGET], scores, name=name, ax=axis)\n        cm_fig, cm_axis = plt.subplots(figsize=(5, 4))\n        ConfusionMatrixDisplay.from_predictions(\n            test[TARGET], scores >= threshold, labels=[0, 1],\n            display_labels=["Legitimate", "Fraud"], ax=cm_axis, colorbar=False)\n        cm_axis.set_title(name)\n        cm_fig.tight_layout()\n        cm_fig.savefig(output / f"confusion_{name}.png", dpi=140)\n        plt.close(cm_fig)\n    axis.axhline(test[TARGET].mean(), linestyle="--", color="gray", label="Prevalence")\n    axis.set_title("Held-out test precision-recall comparison")\n    axis.legend(loc="best")\n    fig.tight_layout()\n    fig.savefig(output / "precision_recall.png", dpi=140)\n    plt.close(fig)\n    validation_table, test_table = pd.DataFrame(rows), pd.DataFrame(test_rows)\n    validation_table.to_csv(output / "validation_metrics.csv", index=False)\n    test_table.to_csv(output / "test_metrics.csv", index=False)\n    test_table.merge(validation_table[["model", "training_seconds", "best_iteration"]],\n                     on="model").to_csv(output / "comparison.csv", index=False)\n    result = state["results"][selected]\n    artifact = {"pipeline": result["pipeline"], "threshold": result["threshold"], "features": FEATURES,\n                    "model_name": selected, "version": state["manifest"]["version"]}\n    dump_model(output / "model.joblib", artifact)\n    sample = test[FEATURES].iloc[:32]\n    reloaded = joblib.load(output / "model.joblib")\n    before = artifact["pipeline"].predict_proba(sample)[:, 1]\n    after = reloaded["pipeline"].predict_proba(sample)[:, 1]\n    np.testing.assert_allclose(before, after, rtol=1e-6, atol=1e-8)\n    np.testing.assert_array_equal(before >= artifact["threshold"], after >= artifact["threshold"])\n    splits = {name: {"rows": len(state[key]), "frauds": int(state[key][TARGET].sum())}\n              for name, key in [("train", "training"), ("validation", "validation"),\n                                ("test", "test"), ("boost_fit", "fit"),\n                                ("boost_stopping", "stopping")]}\n    baseline = evaluate(test[TARGET], np.zeros(len(test)), 0.5)\n    report = {\n        "version": artifact["version"], "selected_model": selected, "mode": "fast",\n        "selection_metric": "validation average_precision",\n        "threshold_policy": "Maximum validation recall at minimum precision; fallback maximum F1.",\n        "minimum_validation_precision": config["min_precision"],\n        "dataset": {**state["summary"], "sha256": config["dataset_sha256"]},\n        "splits": splits, "seed": config["seed"], "jobs": config["jobs"],\n        "split_strategy": "stratified random 70/15/15 after exact-feature deduplication",\n        "stopping_policy": "15% of training only; validation and test excluded from early stopping",\n        "training_devices": {n: config["device"] if n == "xgboost" else "cpu"\n                             for n in config["models"]},\n        "export_device": "cpu", "configuration": config,\n        "preparation_seconds": state["preparation_seconds"],\n        "total_model_fit_seconds": sum(row["training_seconds"] for row in rows),\n        "environment": {"python": platform.python_version(), "platform": platform.platform(),\n                        "packages": config["packages"]},\n        "validation": rows, "test": test_rows, "all_legitimate_baseline": baseline,\n        "limitations": [\n            "Single seeded random split of a historical dataset, not future payment performance.",\n            "Only a small number of frauds are present in the test set; metrics have uncertainty.",\n            "Scores are not calibrated fraud probabilities.",\n            "Validation precision target is not a guarantee on test or future data.",\n            "Upstream PCA fitting procedure is unknown; raw card fields are not supported.",\n            "Batch timing is not HTTP latency or a production capacity claim.",\n        ],\n    }\n    selected_metrics = next(row for row in test_rows if row["model"] == selected)\n    (output / "resume_bullets.md").write_text(resume_text(report, selected_metrics), encoding="utf-8")\n    (output / "requirements-model.txt").write_text(\n        "\\n".join(f"{p}=={v}" for p, v in config["packages"].items()) + "\\n", encoding="utf-8")\n    # Deliberately balanced examples for UI demonstration, never a performance sample.\n    examples = test.groupby(TARGET, sort=True).head(4)\n    write_json(output / "demo_samples.json", {\n        "version": artifact["version"], "note": "Curated held-out examples, not representative traffic.",\n        "examples": [{"label": int(row[TARGET]), "transaction": row[FEATURES].to_dict()}\n                     for _, row in examples.iterrows()],\n    })\n    # Completion marker is written last so interrupted exports can be regenerated.\n    write_json(output / "report.json", report)\n    print(test_table.to_string(index=False))\n    print(f"Selected by validation AP: {selected}. Reload verified. Results: {output.resolve()}")\n    return report\n\n\ndef resume_text(report, metrics):\n    test = report["splits"]["test"]\n    return (\n        "# Measured project results\\n\\n"\n        f"Run: {report[\'version\']}; dataset SHA-256: {report[\'dataset\'][\'sha256\']}\\n\\n"\n        "Use these claims only for the dataset actually used in this run. "\n        "Synthetic test runs are software checks, not resume evidence.\\n\\n"\n        f"- Built a credit-card fraud detection pipeline comparing {len(report[\'test\'])} models "\n        f"on {report[\'dataset\'][\'unique_rows\']:,} deduplicated transactions; selected "\n        f"{report[\'selected_model\']} by validation average precision, achieving "\n        f"{metrics[\'average_precision\']:.3f} test AP, {metrics[\'precision\']:.1%} fraud precision "\n        f"and {metrics[\'recall\']:.1%} fraud recall on a {test[\'rows\']:,}-transaction held-out set "\n        f"({test[\'frauds\']} frauds).\\n"\n        "- Integrated the trained model with FastAPI batch inference and a local dashboard; "\n        "added schema validation, resumable training checkpoints and reproducible evaluation reports.\\n\\n"\n        f"Supporting metrics: accuracy {metrics[\'accuracy\']:.4%}, F1 {metrics[\'f1\']:.4f}, "\n        f"ROC-AUC {metrics[\'roc_auc\']:.4f}; TP={metrics[\'true_positives\']}, "\n        f"FP={metrics[\'false_positives\']}, FN={metrics[\'false_negatives\']}, "\n        f"TN={metrics[\'true_negatives\']}.\\n\\n"\n        f"Predicting every transaction legitimate already gives "\n        f"{report[\'all_legitimate_baseline\'][\'accuracy\']:.4%} accuracy and 0% fraud recall. "\n        "Do not present accuracy alone. These are single-split historical results, "\n        "not production results or a measured training speedup.\\n"\n    )\n\n\ndef train_fast(data, output, **options):\n    state = prepare_run(data, output, **options)\n    for name in state["config"]["models"]:\n        fit_model(state, name)\n    return finish_run(state)\n', 'prediction.py': '"""Trusted artifact loading and bounded-memory batch prediction."""\n\nimport os\nimport tempfile\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_detection.data import FEATURES, TARGET, validate_features\n\n\ndef load_artifact(path: str | Path) -> dict:\n    # Joblib uses pickle: only load artifacts produced by a trusted training process.\n    artifact = joblib.load(path)\n    required = {"pipeline", "threshold", "features", "model_name", "version"}\n    if not isinstance(artifact, dict) or not required.issubset(artifact):\n        raise ValueError("Invalid model artifact.")\n    if artifact["features"] != FEATURES or not 0 <= artifact["threshold"] <= 1:\n        raise ValueError("Incompatible model schema or decision threshold.")\n    return artifact\n\n\ndef score_frame(artifact: dict, frame: pd.DataFrame) -> pd.DataFrame:\n    features = validate_features(frame)\n    scores = artifact["pipeline"].predict_proba(features)[:, 1]\n    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError("Model returned invalid probability scores.")\n    return pd.DataFrame({\n        "fraud_score": scores,\n        "is_fraud": scores >= artifact["threshold"],\n    }, index=frame.index)\n\n\ndef predict_csv(model: str | Path, source: str | Path, output: str | Path,\n                chunk_size: int = 10_000) -> int:\n    if chunk_size < 1:\n        raise ValueError("Chunk size must be positive.")\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if output in (source, Path(model).resolve()):\n        raise ValueError("Prediction output must not overwrite input or model.")\n    artifact = load_artifact(model)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary = tempfile.mkstemp(dir=output.parent, suffix=".csv.tmp")\n    os.close(descriptor)\n    count = 0\n    try:\n        for chunk in pd.read_csv(source, chunksize=chunk_size):\n            if len(chunk) == 0:\n                continue\n            result = score_frame(artifact, chunk.drop(columns=TARGET, errors="ignore"))\n            result.insert(0, "row_number", np.arange(count, count + len(chunk)))\n            result.to_csv(temporary, mode="a", header=count == 0, index=False)\n            count += len(chunk)\n        if not count:\n            raise ValueError("Input CSV has no transactions.")\n        os.replace(temporary, output)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n    return count\n'}
source_root = Path(tempfile.mkdtemp(prefix="fraud-training-"))
package = source_root / "fraud_detection"
package.mkdir()
for name, source in PROJECT_SOURCES.items():
    (package / name).write_text(source, encoding="utf-8", newline="\n")
sys.path.insert(0, str(source_root))
for name in list(sys.modules):
    if name == "fraud_detection" or name.startswith("fraud_detection."):
        del sys.modules[name]
importlib.invalidate_caches()
from fraud_detection.fast_training import prepare_run, fit_model, finish_run
print("Project code ready.")


## 4. Locate dataset, validate GPU, prepare splits
Kaggle download is cached. If authentication is required, run `kagglehub.login()`
interactively; do not store tokens in this notebook.

Exact-feature duplicates are removed before stratified 70/15/15 train/validation/test
splitting. Boosters fit on 85% of **training**, using its remaining 15% for early stopping.
Validation selects model and threshold; test is used only in step 8.
`DEVICE = "auto"` probes XGBoost CUDA support and falls back to CPU when unavailable.
To require GPU training, select a GPU runtime and set `DEVICE = "cuda"`.
If the resolved device differs from a previous run, choose a new `RUN_NAME`.


In [ ]:
if DATA_SOURCE == "kaggle":
    import kagglehub
    DATA_PATH = Path(kagglehub.dataset_download("mlg-ulb/creditcardfraud")) / "creditcard.csv"
elif DATA_SOURCE != "path":
    raise ValueError("DATA_SOURCE must be kaggle or path")

run = prepare_run(DATA_PATH, OUTPUT_DIR, seed=SEED, jobs=JOBS,
                  min_precision=MIN_PRECISION, device=DEVICE, models=MODELS,
                  max_trees=MAX_TREES, patience=PATIENCE, resume=RESUME)


## 5. Logistic Regression
CPU baseline, scaling fitted on training only.
A completed checkpoint is loaded instead of fitting again.

In [ ]:
if "logistic_regression" in MODELS:
    print(fit_model(run, "logistic_regression"))
else:
    print("Skipped logistic_regression")


## 6. Xgboost
T4/CUDA histogram trees; stop after 30 rounds without improvement by default.
A completed checkpoint is loaded instead of fitting again.

In [ ]:
if "xgboost" in MODELS:
    print(fit_model(run, "xgboost"))
else:
    print("Skipped xgboost")


## 7. Lightgbm
Compact CPU histogram booster with early stopping.
A completed checkpoint is loaded instead of fitting again.

In [ ]:
if "lightgbm" in MODELS:
    print(fit_model(run, "lightgbm"))
else:
    print("Skipped lightgbm")


## 8. Freeze selection, evaluate and export
Select by validation AP and select each threshold on validation. Then evaluate the
untouched test split. Accuracy, fraud precision/recall/F1, AP, ROC-AUC and confusion
counts are saved. Re-running a completed export returns saved metrics.

Do not optimize against these test numbers. Compare the all-legitimate baseline:
high accuracy alone is misleading. CPU export and reload predictions are verified.


In [ ]:
import pandas as pd
from IPython.display import Image, display

report = finish_run(run)
display(pd.DataFrame(report["test"]))
display(Image(filename=str(OUTPUT_DIR / "precision_recall.png")))
display(Image(filename=str(OUTPUT_DIR / f"confusion_{report['selected_model']}.png")))
print((OUTPUT_DIR / "resume_bullets.md").read_text(encoding="utf-8"))


## 9. Verify predictions and package only serving files
The archive excludes the raw dataset and training checkpoints. Keep the original
run folder/Drive checkpoints if you want to resume. Package versions are recorded;
verify the artifact in your local environment after transfer.


In [ ]:
import json
import zipfile
import joblib
from fraud_detection.prediction import score_frame
from fraud_detection.data import FEATURES

artifact = joblib.load(OUTPUT_DIR / "model.joblib")  # Only load your own trusted artifacts
display(score_frame(artifact, run["test"][FEATURES].iloc[:5]))
archive = OUTPUT_DIR.with_suffix(".zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT_DIR.iterdir():
        if path.is_file() and not path.name.endswith(".tmp"):
            bundle.write(path, path.name)
print("Download:", archive)
DOWNLOAD_IN_COLAB_BROWSER = False
if DOWNLOAD_IN_COLAB_BROWSER:
    from google.colab import files
    files.download(str(archive))


## 10. Run the model in the project (local PowerShell)
Extract the ZIP into a **new** folder, e.g. `artifacts/colab-fast`, preserving any local run.
Then run from the project root:

```powershell
.\.venv\Scripts\python.exe -m pip install -r artifacts/colab-fast/requirements-model.txt
.\run.ps1 -ModelPath artifacts/colab-fast/model.joblib
```

Open **http://127.0.0.1:8000** for the dashboard and **/docs** for the API.
The dashboard shows the matching report and can score held-out examples or a sample CSV.
For full-dataset CSV inference:

```powershell
.\.venv\Scripts\python.exe -m fraud_detection.cli predict --model artifacts/colab-fast/model.joblib --data data/creditcard.csv --output predictions/scores.csv
```

Use `resume_bullets.md` for the measured values, explaining test size and fraud count.
These are historical random-split results, not real payment deployment evidence.
